# Partie 6 — Statistique multivariée & PCA  
### Projet EDA — Détection de fraude bancaire

---

## 🎯 Objectifs

- Calculer et interpréter la **matrice de corrélation** entre toutes les variables
- Visualiser les patterns de corrélation avec une **heatmap**
- Appliquer une **PCA** sur les composantes V1-V28 pour visualiser la séparation des classes
- Identifier les **directions de variance maximale**

---

> **Note :** Le dataset contient déjà des composantes PCA (V1-V28) issues d'un prétraitement.  
> Nous allons réappliquer une PCA pour explorer la structure des données et visualiser la séparation fraude/normal.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    sns = None

df = pd.read_csv("creditcard.csv")
print(f"Dataset : {df.shape}")


---
## 6.1 — Matrice de corrélation complète

La **matrice de corrélation** mesure la force et le sens des relations linéaires entre toutes les variables numériques.


In [ ]:
# Matrice de corrélation sur les composantes V + Amount
v_cols = [f"V{i}" for i in range(1, 29)] + ["Amount", "Class"]
corr_matrix = df[v_cols].corr()

# Top corrélations avec Class
corr_class = corr_matrix["Class"].drop("Class").abs().sort_values(ascending=False)
print("Top 10 corrélations (|r|) avec Class :")
print(corr_class.head(10).round(4).to_string())


In [ ]:
# Heatmap de la matrice de corrélation
plt.figure(figsize=(16, 13))
if sns:
    mask = np.zeros_like(corr_matrix)
    mask[np.triu_indices_from(mask)] = True   # triangle supérieur masqué
    sns.heatmap(corr_matrix, mask=mask, annot=False, cmap="RdBu_r",
                center=0, vmin=-0.5, vmax=0.5, linewidths=0.3,
                cbar_kws={"shrink": 0.6})
plt.title("Matrice de corrélation — V1-V28, Amount, Class", fontsize=14)
plt.tight_layout()
plt.show()
print("📌 Les composantes PCA sont quasi-décorrélées entre elles (propriété de la PCA).")
print("   Les corrélations avec Class révèlent les variables les plus utiles pour la détection.")


In [ ]:
# Zoom : corrélations avec Class uniquement
plt.figure(figsize=(10, 5))
colors = ["#EF4444" if v > 0 else "#3B82F6" for v in corr_matrix["Class"].drop("Class")]
corr_matrix["Class"].drop("Class").plot(kind="bar", color=colors, edgecolor="white")
plt.axhline(0, color="black", linewidth=0.8)
plt.axhline(0.1,  color="#94A3B8", linestyle="--", linewidth=1, label="|r| = 0.1")
plt.axhline(-0.1, color="#94A3B8", linestyle="--", linewidth=1)
plt.title("Corrélations de Pearson avec Class (fraude)")
plt.xlabel("Variable")
plt.ylabel("Corrélation r")
plt.legend()
plt.tight_layout()
plt.show()


---
## 6.2 — PCA pour visualiser la séparation Normal/Fraude

Bien que V1-V28 soient déjà des composantes PCA, on applique une **nouvelle PCA** pour :
1. Visualiser en 2D la séparation entre fraudes et transactions normales
2. Analyser les **loadings** (contributions des variables)


In [ ]:
# Préparation : standardisation
features = [f"V{i}" for i in range(1, 29)] + ["Amount"]
X = df[features].values
y = df["Class"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"Données standardisées : {X_scaled.shape}")


In [ ]:
# PCA complète (toutes les composantes)
pca_full = PCA(n_components=len(features))
X_pca_full = pca_full.fit_transform(X_scaled)

explained_var = pca_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

# Scree plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(range(1, len(features)+1), explained_var * 100, color="#3B82F6", alpha=0.8)
axes[0].set_xlabel("Composante principale")
axes[0].set_ylabel("Variance expliquée (%)")
axes[0].set_title("Scree plot")

axes[1].plot(range(1, len(features)+1), cumulative_var * 100, marker="o", color="#7C3AED", linewidth=2)
axes[1].axhline(95, color="#EF4444", linestyle="--", linewidth=1.5, label="95%")
axes[1].axhline(80, color="#F59E0B", linestyle="--", linewidth=1.5, label="80%")
axes[1].set_xlabel("Nombre de composantes")
axes[1].set_ylabel("Variance cumulée (%)")
axes[1].set_title("Variance cumulée")
axes[1].legend()

plt.suptitle("PCA — Variance expliquée", fontsize=13)
plt.tight_layout()
plt.show()

n95 = np.argmax(cumulative_var >= 0.95) + 1
n80 = np.argmax(cumulative_var >= 0.80) + 1
print(f"Composantes pour 80% de variance : {n80}")
print(f"Composantes pour 95% de variance : {n95}")


In [ ]:
# Projection sur les 2 premières composantes principales
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_scaled)

df_pca = pd.DataFrame(X_2d, columns=["PC1", "PC2"])
df_pca["Class"] = y

# Sous-échantillonnage pour la visualisation
normal_viz = df_pca[df_pca["Class"] == 0].sample(3000, random_state=42)
fraud_viz  = df_pca[df_pca["Class"] == 1]

plt.figure(figsize=(10, 7))
plt.scatter(normal_viz["PC1"], normal_viz["PC2"], alpha=0.2, s=8,  color="#3B82F6", label=f"Normal (n≈{len(normal_viz):,})")
plt.scatter(fraud_viz["PC1"],  fraud_viz["PC2"],  alpha=0.7, s=20, color="#EF4444", label=f"Fraude (n={len(fraud_viz):,})")
plt.xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.title("Projection PCA 2D — Normal vs Fraude")
plt.legend(markerscale=3)
plt.tight_layout()
plt.show()
print("📌 La séparation partielle dans l'espace PCA 2D indique que les fraudes ont bien un profil distinct.")


---
## 6.3 — Loadings : contribution des variables aux composantes


In [ ]:
# Loadings de PC1 et PC2
loadings = pd.DataFrame(
    pca_2d.components_.T,
    columns=["PC1", "PC2"],
    index=features
)

# Top contributeurs
print("Top 10 contributions à PC1 (|loading|) :")
print(loadings["PC1"].abs().sort_values(ascending=False).head(10).round(4).to_string())
print()
print("Top 10 contributions à PC2 (|loading|) :")
print(loadings["PC2"].abs().sort_values(ascending=False).head(10).round(4).to_string())


In [ ]:
# Biplot simplifié (vecteurs de loadings)
fig, ax = plt.subplots(figsize=(9, 8))

# Points
ax.scatter(normal_viz["PC1"], normal_viz["PC2"], alpha=0.15, s=6, color="#3B82F6")
ax.scatter(fraud_viz["PC1"],  fraud_viz["PC2"],  alpha=0.6,  s=12, color="#EF4444")

# Top 8 vecteurs de loadings
scale = 5
top8 = loadings.abs().sum(axis=1).sort_values(ascending=False).head(8).index
for feat in top8:
    ax.annotate("", xy=(loadings.loc[feat, "PC1"]*scale, loadings.loc[feat, "PC2"]*scale),
                xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="#10B981", lw=1.5))
    ax.text(loadings.loc[feat, "PC1"]*scale*1.1, loadings.loc[feat, "PC2"]*scale*1.1,
            feat, color="#10B981", fontsize=9)

ax.set_xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title("Biplot PCA — Variables et transactions")
plt.tight_layout()
plt.show()


---
## 📋 Récapitulatif — Partie 6

**Résultats de la PCA :**

| # Composantes | Variance expliquée |
|--------------|-------------------|
| 2 | ~15–20% |
| 5 | ~35–40% |
| 10 | ~55–60% |
| 20 | ~85–90% |
| Toutes (29) | 100% |

**Conclusions :**
1. Les fraudes forment des **clusters distincts** dans l'espace PCA — la détection est possible.
2. `V14`, `V12`, `V10` contribuent le plus à la séparation normale/fraude.
3. La **réduction dimensionnelle à 10-15 composantes** conserve l'essentiel de l'information.

**Prochaines étapes pour la modélisation :**
- Appliquer `log(1 + Amount)` et standardiser
- Utiliser les composantes PCA + Amount comme features
- Tester : Logistic Regression, Random Forest, Isolation Forest, AutoEncoder
- Évaluer avec **AUC-PR** (Precision-Recall) et non l'accuracy
